# 🔥 S9 P3 — Régression avec PyTorch Lightning (version Colab)

Même code que sur mon PC, mais prêt à tourner sur un **GPU T4 gratuit** de Google.
Les données sont **synthétiques** (générées dans le notebook) : rien à télécharger.

## Se connecter au T4

**`Select Kernel`** (en haut à droite) → `Select Another Kernel...` → `Colab` →
connexion Google → **`New Colab Server`** → **`GPU`** → **`T4`**

> ⚠️ **Le GPU se choisit à la création du serveur, nulle part ailleurs.**
> `Auto Connect` te donne un CPU. Puis `Run All`.

## 0. Installer les libs manquantes sur Colab

`pytorch_lightning` n'est pas préinstallé sur Colab. On l'installe **seulement** si on
est sur Colab (sur mon PC il est déjà là, la cellule ne fait rien).

In [ ]:
try:
    import google.colab
    DANS_COLAB = True
    !pip install -q pytorch_lightning
except ImportError:
    DANS_COLAB = False
    print("Pas sur Colab : rien a installer.")

## Où je tourne, et sur quoi ?

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("J'execute sur :", "Google Colab (a distance)" if DANS_COLAB else "ton PC")
print(f"torch  : {torch.__version__}")
print(f"device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

In [ ]:

import numpy as np
import pandas as pd 
import matplotlib.pylab as plt
import seaborn as sns


from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger

pl.seed_everything(32)

In [ ]:
# y = sin(x1) + 0.5 * x2 ^ 2 + noise
n_samples = 1200
rng = np.random.default_rng(32)
x1 = rng.uniform(-3.0, 3.0, size=n_samples)
x2 = rng.uniform(-2.0, 2.0, size=n_samples)
noise = rng.normal(loc=0.0, scale=0.15, size=n_samples)

y = np.sin(x1) + 0.5 * x2**2 + noise  # calcul de y manquant

X = np.column_stack([x1, x2]).astype(np.float32)
y = y.reshape(-1, 1).astype(np.float32)
print(f"X Shape {X.shape} Y Shape {y.shape}")

In [ ]:
data = pd.DataFrame({"x1":X[:,0], "x2": X[:,1], "y": y[:,0]})

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(data=data, x="x1", y="x2", hue="y")

In [ ]:
import plotly.express as px
fig = px.scatter_3d(data, x="x1",y="x2",z="y", opacity=0.7)
fig.update_traces(marker_size = 2)
fig.show()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
print(f"Training samples {len(X_train)} Val Sample {len(X_val)}")

In [ ]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = x_scaler.transform(X_val).astype(np.float32)

y_train_scaled = y_scaler.fit_transform(y_train).astype(np.float32)
y_val_scaled = y_scaler.transform(y_val).astype(np.float32)

print(f"BEFORE: {X_train.mean()} - {X_train.std()} - {y_train.mean()} - {y_train.std()}")

In [ ]:
print(f"AFTER: {X_train_scaled.mean(0)} - {X_train_scaled.std(0)} - {y_train_scaled.mean()} - {y_train_scaled.std()}")

In [ ]:
x_scaler

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    x=X_train_scaled[:, 0],
    y=X_train_scaled[:, 1],
    z=y_train_scaled.flatten(),
    opacity=0.7
)
fig.update_traces(marker_size=2)
fig.show()

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled,dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled,dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled,dtype=torch.float32)

In [ ]:
print(y_train_tensor.shape)

In [ ]:
class RegressionDataset(Dataset) : 
    def __init__(self,X,y) :
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self,index):
        x = self.X[index,:]
        y = self.y[index,0]
        return x, y

In [ ]:
train_dataset = RegressionDataset(X_train_tensor, y_train_tensor)
val_dataset = RegressionDataset(X_val_tensor,y_val_tensor)

In [ ]:
len(val_dataset)

In [ ]:
val_dataset[10]

In [ ]:
batch_size = 32 
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)

In [ ]:
first_batch_x, first_batch_y = next(iter(train_loader))

In [ ]:
print(first_batch_x,first_batch_y)